In [1]:
from pathlib import Path
import dask.array as da
import pandas as pd
import anndata as ad
import napari
import numpy as np
from tqdm.auto import tqdm
import json
import seaborn as sns
import matplotlib.pyplot as plt
expanded_piyg = ['#1a9641', '#a6d96a', '#978897', '#d1d1ca', '#f1b6da', '#d02c91']


def timed_compute(volume):
    """
    Compute a lazy Dask array frame-by-frame with progress reporting.

    This function iterates over the leading axis of a Dask array (e.g. time),
    calls `.compute()` on each slice, and stacks the results into a single
    NumPy array. A tqdm progress bar is displayed to indicate progress.

    Parameters
    ----------
    volume : dask.array.Array
        A Dask array with at least one dimension (e.g. shape (T, ...)).
        The function will iterate over the first axis (axis=0).

    Returns
    -------
    numpy.ndarray
        A NumPy array with the same shape as `volume`, but fully realized
        in memory. The dtype is preserved from the Dask array.

    Notes
    -----
    - Each frame is computed independently, which can be helpful for
      monitoring performance and memory use on large arrays.
    - The returned array may be very large if `volume` is large.
      Ensure sufficient memory is available.
    """
    return np.stack([frame.compute() for frame in tqdm(volume)], axis=0)


In [3]:
root_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/Z_stack_tests/zarr')
# root_dir = Path('/Volumes/OPERA3/Nathan/Z_stack_tests/zarr')
rc_stem = "(3,3)"
store = root_dir / f"{rc_stem}.zarr"

# --- load images and segmentation ---
images = da.from_zarr(str(store / "images" / "0"))   # (T,C,Z,Y,X)
masks  = da.from_zarr(str(store / "labels" / "masks"))  # (T,Z,Y,X)
tracked_masks = da.from_zarr(str(store / "labels" / "masks_tracked"))  # (T,Z,Y,X)
# --- load tracks (CSV preferred, fallback to AnnData) ---
# tracks_csv = root_dir / f"{rc_stem}_tracks.csv"
# if tracks_csv.exists():
#     tracks = pd.read_csv(tracks_csv)
# else:
adata = ad.read_zarr(str(store / "tables" / "quantified_tracks"))
tracks = adata.obs.reset_index(drop=True)

print("images:", images.shape)
print("masks:", masks.shape)
print("tracked masks:", tracked_masks.shape)
print("tracks:", tracks.shape)


images: (97, 2, 25, 7992, 7992)
masks: (97, 25, 7992, 7992)
tracked masks: (97, 7992, 7992)
tracks: (1859520, 10)


In [4]:
images

dask.array<from-zarr, shape=(97, 2, 25, 7992, 7992), dtype=uint16, chunksize=(1, 1, 16, 512, 512), chunktype=numpy.ndarray>

In [5]:
tracked_masks

dask.array<from-zarr, shape=(97, 7992, 7992), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

In [16]:
tracks = tracks[tracks['z'] == 12]
tracks

,t,ID,z,y,x,cell_area_px,mtb_area_px,segment_ID,row,col
12,0,1,12,866,6511,13475,0,817,3,3
29,0,6,12,525,4880,14200,0,755,3,3
47,0,9,12,7490,1929,11750,0,64,3,3
65,0,10,12,4221,542,21675,0,1009,3,3
83,0,11,12,4548,823,14975,18,442,3,3
...,...,...,...,...,...,...,...,...,...,...
1859434,96,8451,12,6487,2633,16700,0,361,3,3
1859449,96,8492,12,6519,109,11050,0,1758,3,3
1859470,96,8531,12,5452,1006,12250,69,646,3,3
1859487,96,8562,12,7760,1892,17300,215,1438,3,3


In [17]:
napari_tracks = tracks[['ID', 't', 'z', 'y', 'x']].to_numpy(dtype=np.int64)
napari_tracks.shape

(108600, 5)

In [9]:
viewer = napari.Viewer(title = 'inspecting tracks and quantifications')
viewer.add_image(images, channel_axis=1)
viewer.add_labels(masks)
viewer.add_tracks(napari_tracks)

<Tracks layer 'napari_tracks' at 0x7534686b68c0>

In [12]:
napari_tracks[:, 1].shape

(1859520,)

In [19]:
# features = {
#     'time': tracks['t'].values,
#     'mtb_area_px': tracks['mtb_area_px'].values
# }

features = {
    col: tracks[col].values
    for col in tracks.columns
}

In [20]:
viewer.layers['napari_tracks'].features = features

In [21]:
scale = [1.0, 0.14949, 0.14949]

In [22]:
for layer in viewer.layers:
    layer.scale = scale

In [14]:
import napari_animation

In [25]:
def update_slider(event):
    # Compute time in hours
    time = viewer.dims.current_step[0] / 2
    # Update the text overlay (bottom left)
    viewer.text_overlay.text = f"{time:1.2f} hrs"

# Configure text overlay
viewer.text_overlay.visible = True
viewer.text_overlay.color = "white"
viewer.text_overlay.font_size = 24
viewer.text_overlay.position = "bottom_left"

# Add scale bar (bottom right)
viewer.scale_bar.visible = True
viewer.scale_bar.unit = "µm"
viewer.scale_bar.ticks = False
viewer.scale_bar.colored = False
viewer.scale_bar.font_size = 24
viewer.scale_bar.color = "white"
viewer.scale_bar.position = "bottom_right"

# Connect slider update
viewer.dims.events.current_step.connect(update_slider)


<function __main__.update_slider(event)>

In [26]:
viewer.camera

Camera(center=(12.0, 597.287295, 597.287295), zoom=0.7577900329923872, angles=(0.490555016764744, 9.421260731998562, -90.20087752517617), perspective=0.0, mouse_pan=True, mouse_zoom=True)

In [ ]:
from napari_animation import Animation
from tqdm.auto import tqdm
import numpy as np

# --- Movie 1: straight time-through ---
viewer.dims.ndisplay = 3

anim1 = Animation(viewer)

nT = viewer.dims.nsteps[0]
fps = 30
duration_s = 20
total_frames = int(round(duration_s * fps))
steps_between = max(1, total_frames // max(1, (nT - 1)))

# fix camera
cam = viewer.camera
cam_center, cam_zoom, cam_angles, cam_persp = tuple(cam.center), cam.zoom, tuple(cam.angles), cam.perspective

step0 = list(viewer.dims.current_step)
step0[0] = 0
viewer.dims.current_step = tuple(step0)
viewer.camera.center = cam_center
viewer.camera.zoom = cam_zoom
viewer.camera.angles = cam_angles
viewer.camera.perspective = cam_persp
anim1.capture_keyframe()

for t in tqdm(range(1, nT), desc="Capturing time keyframes"):
    step = list(step0)
    step[0] = t
    viewer.dims.current_step = tuple(step)
    anim1.capture_keyframe(steps=steps_between)

anim1.animate("timelapse_time.mp4", fps=fps, canvas_only=True, size=(1920, 1080))


# --- Movie 2: oblique tilt + zoom + pan ---
anim2 = Animation(viewer)

# baseline camera
cam0 = viewer.camera
c0 = np.array(cam0.center, dtype=float)
z0 = float(cam0.zoom)
a0 = np.array(cam0.angles, dtype=float)
p0 = float(cam0.perspective)

# targets
c1 = c0 + np.array([100.0, -100.0, 0.0])
z1 = z0 * 2.0
a1 = a0 + np.array([20.0, 45.0, 0.0])
p1 = p0

step0 = list(viewer.dims.current_step)
step0[0] = 0
viewer.dims.current_step = tuple(step0)

# initial
viewer.camera.center = tuple(c0)
viewer.camera.zoom = z0
viewer.camera.angles = tuple(a0)
viewer.camera.perspective = p0
anim2.capture_keyframe()

for t in tqdm(range(1, nT), desc="Capturing oblique keyframes"):
    u = t / (nT - 1)  # 0→1
    step = list(step0)
    step[0] = t
    viewer.dims.current_step = tuple(step)

    viewer.camera.center = tuple(c0 * (1 - u) + c1 * u)
    viewer.camera.zoom = z0 * (1 - u) + z1 * u
    viewer.camera.angles = tuple(a0 * (1 - u) + a1 * u)
    viewer.camera.perspective = p0 * (1 - u) + p1 * u

    anim2.capture_keyframe(steps=steps_between)

anim2.animate("timelapse_oblique.mp4", fps=fps, canvas_only=True, size=(1920, 1080))


Capturing time keyframes:   0%|          | 0/96 [00:00<?, ?it/s]